# Jalon 1 - Cluster, RDD et Spark SQL

**Public :** binôme du module Écosystème Big Data 2. **Prérequis :** Docker Compose démarré, notions de Python et SQL.

Ce notebook exécute les scripts du dépôt et affiche les résultats essentiels. Ouvrir les fichiers `src/part1_cluster.py`, `src/part2_rdd.py` et `src/part3_sql.py` pour étudier les transformations.

1. Préparer les données et le stockage ; 2. vérifier le cluster ; 3. analyser en RDD ; 4. comparer DataFrames/SQL ; 5. contrôler les résultats.

Les données sont un **remplacement synthétique autorisé**, pas les fichiers officiels de l'enseignant. Les analyses comptent les événements DELIVERED, doublons inclus ; le dédoublonnage streaming appartient au jalon 2.

In [1]:
from pathlib import Path
import json, subprocess, sys
from IPython.display import display, Markdown
ROOT = Path('/workspace')
def read_result(name):
    return json.loads((ROOT / 'results' / name).read_text(encoding='utf-8'))
def run_script(name, *args):
    completed = subprocess.run([sys.executable, str(ROOT / 'src' / name), *args], cwd=ROOT, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    (ROOT / 'results' / (name + '.log')).write_text(completed.stdout, encoding='utf-8')
    if completed.returncode:
        print(completed.stdout[-12000:])
        raise RuntimeError(f'{name} a échoué')
    print(f'{name} : OK')
def table(rows):
    keys = list(rows[0])
    display(Markdown('| ' + ' | '.join(keys) + ' |\n| ' + ' | '.join(['---'] * len(keys)) + ' |\n' + '\n'.join('| ' + ' | '.join(str(r[k]) for k in keys) + ' |' for r in rows)))

## Données et ressources
Graine fixe 2026, 50 000 événements, 200 restaurants, 2 % de doublons exacts, environ 5 % de retards jusqu'à 15 minutes. Le bootstrap crée le bucket `lake` et le topic `orders` (3 partitions), puis copie les fichiers dans Bronze.

In [2]:
run_script('producer.py', '--mode', 'file', '--n', '50000')
run_script('bootstrap.py')
read_result('bootstrap.json')

producer.py : OK


bootstrap.py : OK


{'bucket': 'lake',
 'topic': 'orders',
 'partitions': 3,
 'objects': [{'key': 'bronze/history/generation_manifest.json', 'bytes': 629},
  {'key': 'bronze/history/orders_history.jsonl', 'bytes': 12287363},
  {'key': 'bronze/reference/restaurants.json', 'bytes': 30634}]}

## Partie 1 - Cluster
L'action de somme s'exécute sur le cluster. Deux Workers doivent être actifs. `defaultParallelism` est un nombre de partitions par défaut : on affiche séparément les cœurs déclarés par les Workers.

In [3]:
run_script('part1_cluster.py')
read_result('part1_cluster.json')

part1_cluster.py : OK


{'spark_version': '3.5.6',
 'master': 'spark://spark-master:7077',
 'app_name': 'Jalon1-Partie1-Cluster',
 'application_id': 'app-20261106113541-0000',
 'default_parallelism': 4,
 'python': '3.8.10',
 'alive_workers': 2,
 'worker_cores': 4,
 'test_sum': 499500,
 'note': 'defaultParallelism is a default number of partitions, not a CPU inventory.'}

## Partie 2 - RDD
`textFile`, `map`, `filter` et `reduceByKey` construisent un calcul différé. `collect`, `count` et `takeOrdered` déclenchent son exécution. Le panier moyen utilise un couple (somme, nombre), agrégé en une seule réduction.

In [4]:
run_script('part2_rdd.py')
p2 = read_result('part2_rdd.json')
table(p2['cities'])
table(p2['top5_restaurants'])
print((ROOT / 'results/part2_lineage.txt').read_text())

part2_rdd.py : OK


| city | revenue_cents | revenue_mad | delivered_events | average_mad |
| --- | --- | --- | --- | --- |
| Casablanca | 56036022 | 560360.22 | 2997 | 186.9737137137137 |
| Fes | 57725665 | 577256.65 | 3100 | 186.21182258064516 |
| Marrakech | 58013675 | 580136.75 | 3106 | 186.7793786220219 |
| Rabat | 57218657 | 572186.57 | 3064 | 186.7449640992167 |
| Tanger | 56275173 | 562751.73 | 2992 | 188.0854712566845 |

| restaurant_id | delivered_events |
| --- | --- |
| R-032 | 101 |
| R-070 | 97 |
| R-059 | 95 |
| R-187 | 95 |
| R-097 | 94 |

(4) PythonRDD[11] at collect at /workspace/src/part2_rdd.py:18 []
 |  MapPartitionsRDD[10] at mapPartitions at PythonRDD.scala:160 []
 |  ShuffledRDD[9] at partitionBy at <unknown>:0 []
 +-(4) PairwiseRDD[8] at reduceByKey at /workspace/src/part2_rdd.py:17 []
    |  PythonRDD[7] at reduceByKey at /workspace/src/part2_rdd.py:17 []
    |  s3a://lake/bronze/history/orders_history.jsonl MapPartitionsRDD[1] at textFile at <unknown>:0 []
    |  s3a://lake/bronze/history/orders_history.jsonl HadoopRDD[0] at textFile at <unknown>:0 []


## Partie 3 - DataFrames et SQL
Schémas explicites, jointure broadcast du référentiel de 200 lignes, comparaison UDF/native, export Parquet partitionné par `city` et relecture. Les mesures comprennent une chauffe puis trois répétitions avec ordre alterné. Une action consomme la colonne calculée pour empêcher Spark d'éliminer l'UDF du test.

In [5]:
run_script('part3_sql.py')
p3 = read_result('part3_sql.json')
table([{'API': name, 'médiane (s)': round(value['median_seconds'], 4)} for name, value in p3['benchmarks'].items()])
table(p3['revenue_by_category'])
print('Lignes Silver relues :', p3['silver_rows'])
print('Stratégie :', p3['join_strategy'])

part3_sql.py : OK


| API | médiane (s) |
| --- | --- |
| RDD | 0.7161 |
| DataFrame | 0.5343 |
| SQL | 0.6108 |
| UDF | 0.9383 |
| Native | 0.6552 |

| city | category | revenue_mad |
| --- | --- | --- |
| Casablanca | burger | 112716.08 |
| Casablanca | marocain | 116921.72 |
| Casablanca | pizza | 114729.30 |
| Casablanca | sushi | 100013.67 |
| Casablanca | tacos | 115979.45 |
| Fes | burger | 115291.72 |
| Fes | marocain | 117729.42 |
| Fes | pizza | 121641.97 |
| Fes | sushi | 118309.95 |
| Fes | tacos | 104283.59 |
| Marrakech | burger | 120508.70 |
| Marrakech | marocain | 117279.99 |
| Marrakech | pizza | 107086.31 |
| Marrakech | sushi | 111384.25 |
| Marrakech | tacos | 123877.50 |
| Rabat | burger | 115266.10 |
| Rabat | marocain | 111478.93 |
| Rabat | pizza | 114593.88 |
| Rabat | sushi | 112516.83 |
| Rabat | tacos | 118330.83 |
| Tanger | burger | 108976.15 |
| Tanger | marocain | 113124.63 |
| Tanger | pizza | 117143.42 |
| Tanger | sushi | 107363.02 |
| Tanger | tacos | 116144.51 |

Lignes Silver relues : 50000
Stratégie : BroadcastHashJoin (explicit broadcast of 200 restaurants)


## Plan de jointure
Les sections Parsed/Analyzed décrivent le plan logique ; Optimized le travail de Catalyst ; Physical la stratégie exécutée. Repérer `BroadcastHashJoin` et `BroadcastExchange`.

In [6]:
print((ROOT / 'results/part3_explain_join.txt').read_text())

== Parsed Logical Plan ==
'Join UsingJoin(Inner, [restaurant_id])
:- Relation [order_id#0,restaurant_id#1,customer_id#2,city#3,status#4,amount_mad#5,items_count#6,distance_km#7,courier_id#8,event_time#9] json
+- ResolvedHint (strategy=broadcast)
   +- Project [restaurant_id#20, name#21 AS restaurant_name#100, restaurant_city#93, category#23, latitude#24, longitude#25]
      +- Project [restaurant_id#20, name#21, city#22 AS restaurant_city#93, category#23, latitude#24, longitude#25]
         +- Relation [restaurant_id#20,name#21,city#22,category#23,latitude#24,longitude#25] json

== Analyzed Logical Plan ==
restaurant_id: string, order_id: string, customer_id: string, city: string, status: string, amount_mad: decimal(12,2), items_count: int, distance_km: double, courier_id: string, event_time: timestamp, restaurant_name: string, restaurant_city: string, category: string, latitude: double, longitude: double
Project [restaurant_id#1, order_id#0, customer_id#2, city#3, status#4, amount_mad

## Validation indépendante
Un calcul Python standard vérifie les montants en centimes, les paniers moyens, le top 5 et les indicateurs par catégorie. Les journaux Spark fournissent les vrais jobs et stages. Les liens History Server restent consultables après la fin des applications.

In [7]:
run_script('validate_results.py')
run_script('collect_evidence.py')
display(read_result('validation.json'))
metrics = read_result('execution_metrics.json')
table([{'partie': part, 'jobs': value['number_of_jobs'], 'stages terminés': value['number_of_completed_stage_attempts'], 'interface': value['history_url']} for part, value in metrics.items()])

validate_results.py : OK


collect_evidence.py : OK


{'status': 'PASS',
 'oracle': 'Python standard library, independent of Spark',
 'checks': ['50000 events and 200 restaurants',
  '1000 exact order/status duplicate rows',
  'Referential integrity and city consistency',
  'RDD = DataFrame = SQL = independent revenue totals',
  'Counts and average baskets',
  'Top 5 restaurants with deterministic tie breaking',
  'Category/city revenue',
  'Distance band counts',
  'Parquet read-back count and revenue (Spark assertions)'],
 'delivered_events': 15259,
 'revenue_mad': 2852691.92}

| partie | jobs | stages terminés | interface |
| --- | --- | --- | --- |
| part1 | 1 | 1 | http://localhost:18080/history/app-20261106113541-0000/jobs/ |
| part2 | 5 | 8 | http://localhost:18080/history/app-20261106113557-0001/jobs/ |
| part3 | 72 | 76 | http://localhost:18080/history/app-20261106113619-0002/jobs/ |

## Exercice de préparation à la soutenance
Pourquoi `orders.withColumn('band', udf(...)).count()` ne suffit-il pas à mesurer le coût de l'UDF ? Quelle différence entre le nombre de Workers et le nombre d'Executors ?

**Réponse guidée :** Catalyst peut supprimer la colonne inutilisée ; il faut consommer sa valeur (ici groupBy). Un Worker est un service du cluster ; un Executor est un processus propre à une application, lancé sur un Worker.

**Piège courant :** les événements d'une même commande comportent plusieurs statuts. Sommer tous les statuts gonflerait le chiffre d'affaires. Ici le filtre DELIVERED est commun aux trois API, mais les doublons sont encore conservés.

**Extension :** comparer ensuite les résultats après dédoublonnage `(order_id, status)` et quantifier l'écart, sans remplacer les résultats exigés des parties 2/3.